![logo](../.././docs/images/Logo_Destination_Earth_Colours.png)

# Exercise 1: Basic Data Retrieval from Climate DT

In this notebook we will pull basic full-field data from the **Climate Digital Twin**. The Climate DT provides climate projection data on a HEALPix grid, which we will retrieve, inspect, plot, and convert to xarray.

## Authentication

In [ ]:
%%capture cap
%run ../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [24]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
import os

## Live Request Toggle

In [25]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

True

## Build the Request

We request 2m temperature from the Climate DT projections dataset.

**EXERCISE**: Change the `date` to a different date in 2040 and add parameter `165` (10m U wind) to the request.

In [26]:
request = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400102",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167",  # EXERCISE: Add parameter 165 (10m U wind) here as well
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "time": "1200",
    "type": "fc",
}

## Retrieve the Data

In [ ]:
data_file = "../climate-dt/data/climate-dt-basic-training.grib"

if LIVE_REQUEST:
    data = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data.to_target("file", data_file)
else:
    data = earthkit.data.from_source("file", data_file)

## Inspect the Data

Use `.ls()` to inspect the fields returned.

In [28]:
data.ls()

,centre,shortName,typeOfLevel,level,dataDate,dataTime,stepRange,dataType,number,gridType
0,ecmf,2t,heightAboveGround,2,20400102,1200,0,fc,None,healpix


## Plot the Data

**EXERCISE**: Complete the plotting code below to visualize the first field. Add coastlines and a title.

In [ ]:
chart = earthkit.plots.Map(extent=[-180, 180, -90, 90])
chart.grid_cells(data, style="auto")

# EXERCISE: Add coastlines, gridlines, and a descriptive title
#chart.coastlines()
#chart.gridlines()
#chart.title("___")

chart.show()

## Convert to xarray

To work with the data in xarray, we first regrid from HEALPix to a regular lat-lon grid.

In [ ]:
data_latlon = earthkit.regrid.interpolate(data, out_grid={"grid": [1, 1]}, method="linear")
ds = data_latlon.to_xarray()
ds